# 🔗 Chapter 3: Conditional Probability and Independence
## MAT204 — Probability and Statistics for Engineers

**Topics:**
1. Conditional Probability — Definition and Examples
2. The Multiplication Rule — Chaining Probabilities
3. The Law of Total Probability
4. Bayes' Theorem
5. Independent Events
6. Mutual Independence
7. Probability Perception and Cognitive Biases (The Linda Problem)

---

In [ ]:
import math
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, FancyArrow
from fractions import Fraction

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("Libraries loaded ✓")

---
## 1. Conditional Probability

**Definition:** For two events $A$ and $B$ with $P(B) > 0$, the conditional probability of $A$ given that $B$ has occurred is:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Intuition (equally likely space):** $P(A \mid B) = \dfrac{\#(A \cap B)}{\#(B)}$

That is, we treat $B$ as the new sample space and compute the proportion of $A$ within that set.

In [ ]:
def conditional_prob(p_A_and_B, p_B):
    """P(A|B) = P(A∩B) / P(B)"""
    assert p_B > 0, "P(B) must be > 0!"
    return p_A_and_B / p_B

def conditional_uniform(S, A, B):
    """P(A|B) = |A∩B| / |B| for an equally likely sample space"""
    intersection = A & B
    return len(intersection) / len(B) if len(B) > 0 else 0

# ──────────────────────────────────────────
# Example 1: Rolling a 1 given an odd number
S_die = set(range(1, 7))
A = {1}              # A = {rolling a 1}
B = {1, 3, 5}        # B = {odd number}

p_A   = len(A) / len(S_die)
p_B   = len(B) / len(S_die)
p_AnB = len(A & B) / len(S_die)
p_AgB = conditional_uniform(S_die, A, B)

print("Example 1: Die — Rolling a 1 given an odd number")
print(f"  S = {sorted(S_die)}")
print(f"  A = {sorted(A)}  (rolling a 1),   P(A) = {p_A:.4f} = {Fraction(len(A),len(S_die))}")
print(f"  B = {sorted(B)}  (odd number),     P(B) = {p_B:.4f} = {Fraction(len(B),len(S_die))}")
print(f"  A∩B = {sorted(A&B)},  P(A∩B) = {p_AnB:.4f} = {Fraction(len(A&B),len(S_die))}")
print(f"  P(A|B) = P(A∩B)/P(B) = {p_AnB:.4f}/{p_B:.4f} = {p_AgB:.4f} = {Fraction(len(A&B),len(B))}")
print(f"  Interpretation: We KNOW an odd number occurred → selection from {{1,3,5}} only")

print()

# Example 2: Two dice — sum = 7, given first die = 4
S_two = [(i,j) for i in range(1,7) for j in range(1,7)]
A2 = {(i,j) for i,j in S_two if i+j == 7}   # Sum = 7
B2 = {(i,j) for i,j in S_two if i == 4}      # First die = 4

p_A2gB2 = conditional_uniform(set(S_two), A2, B2)

print("Example 2: Two Dice — Sum = 7, given first die = 4")
print(f"  A = {{Sum = 7}} = {sorted(A2)}")
print(f"  B = {{First die = 4}} = {sorted(B2)}")
print(f"  A∩B = {sorted(A2&B2)}  ← only (4,3)")
print(f"  P(A|B) = {len(A2&B2)}/{len(B2)} = {p_A2gB2:.4f} = {Fraction(len(A2&B2),len(B2))}")

In [ ]:
# Conditional probability visualization — shrinking of the sample space
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: full die sample space
ax = axes[0]
ax.set_aspect('equal'); ax.set_title('Unconditional: P(A=1) = 1/6', fontsize=12, fontweight='bold')
ax.set_xlim(0, 7); ax.set_ylim(0, 2)
ax.axis('off')
for i, n in enumerate(range(1, 7)):
    color = '#e41a1c' if n == 1 else '#cccccc'
    c = Circle((i+0.8, 1), 0.38, color=color, alpha=0.8, zorder=2)
    ax.add_patch(c)
    ax.text(i+0.8, 1, str(n), ha='center', va='center', fontsize=14,
            fontweight='bold', color='white' if n==1 else 'black')
ax.text(3.5, 0.2, 'S = {1,2,3,4,5,6}  →  P(1) = 1/6', ha='center', fontsize=10)

# Right: reduced sample space given B={odd}
ax2 = axes[1]
ax2.set_aspect('equal'); ax2.set_title('Conditional: P(A=1 | B=odd) = 1/3', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 7); ax2.set_ylim(0, 2)
ax2.axis('off')
for i, n in enumerate([1, 3, 5]):
    color = '#e41a1c' if n == 1 else '#4daf4a'
    c = Circle((i*2+1, 1), 0.38, color=color, alpha=0.85, zorder=2)
    ax2.add_patch(c)
    ax2.text(i*2+1, 1, str(n), ha='center', va='center', fontsize=14,
             fontweight='bold', color='white')
# Faded (excluded)
for i, n in enumerate([2, 4, 6]):
    c = Circle((i*2+0.3, 0.3), 0.22, color='#dddddd', alpha=0.5, zorder=1)
    ax2.add_patch(c)
    ax2.text(i*2+0.3, 0.3, str(n), ha='center', va='center', fontsize=9, color='#aaaaaa')
ax2.text(3.5, 1.7, 'B = {odd} is the new sample space', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#ffffcc'))
ax2.text(3.5, 0.1, 'P(1|odd) = 1/3', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#ffcccc'))

plt.suptitle('Conditional Probability — Reduction of the Sample Space',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Properties of conditional probability — numerical verification
print("Propositions of Conditional Probability")
print("=" * 45)

S = set(range(1, 7))
A = {2, 4, 6};  B = {1, 2, 3, 4}

def p(E, space=S):  return len(E & space) / len(space)

# Proposition 1: P(A|A) = 1
p_AgA = conditional_uniform(S, A, A)
print(f"1. P(A|A) = {p_AgA:.4f}  {'✓' if abs(p_AgA-1)<1e-9 else '✗'}")

# Proposition 2: P(Aᶜ|A) = 0
Ac = S - A
p_AcgA = conditional_uniform(S, Ac, A)
print(f"2. P(Aᶜ|A) = {p_AcgA:.4f}  {'✓' if abs(p_AcgA)<1e-9 else '✗'}")

# Proposition 3: P(Aᶜ|B) = 1 - P(A|B)
p_AgB  = conditional_uniform(S, A, B)
p_AcgB = conditional_uniform(S, Ac, B)
print(f"3. P(Aᶜ|B) = {p_AcgB:.4f},  1-P(A|B) = {1-p_AgB:.4f}  {'✓' if abs(p_AcgB-(1-p_AgB))<1e-9 else '✗'}")
print()

# Conditional probability table — all pairs of dice
S_two = [(i,j) for i in range(1,7) for j in range(1,7)]
print("P(Sum=k | First die = i) matrix:")
print(f"{'':>4}", end="")
for k in range(2, 13):
    print(f"{k:>6}", end="")
print()
print("-" * 72)

for i in range(1, 7):
    B_first = {(a, b) for a, b in S_two if a == i}
    print(f"i={i} |", end="")
    for k in range(2, 13):
        A_sum = {(a,b) for a,b in S_two if a+b==k}
        pval = len(A_sum & B_first) / len(B_first)
        print(f"  {pval:.2f}", end="")
    print()

---
## 2. The Multiplication Rule — Chaining Probabilities

$$P(A \cap B) = P(A \mid B) \cdot P(B)$$

**Generalization (Chain Rule):**
$$P\!\left(\bigcap_{i=1}^n A_i\right) = P(A_1)\cdot P(A_2\mid A_1)\cdot P(A_3\mid A_1,A_2)\cdots P(A_n\mid A_1,\ldots,A_{n-1})$$

In [ ]:
# Urn example — sampling without replacement
# 8 White (W) + 4 Black (B) balls, 3 draws

def multiplication_rule_balls(n_white, n_black, sequence):
    """
    sequence: e.g. ['W','B','B'] → W=White, B=Black
    Sampling without replacement — multiplication rule
    """
    n_total = n_white + n_black
    n_w, n_b = n_white, n_black
    probability = 1.0
    steps = []
    for color in sequence:
        if color == 'W':
            p = n_w / n_total
            steps.append((f'P(W|current:{n_w}W,{n_b}B)', p, Fraction(n_w, n_total)))
            n_w -= 1
        else:   # 'B' (Black)
            p = n_b / n_total
            steps.append((f'P(B|current:{n_w}W,{n_b}B)', p, Fraction(n_b, n_total)))
            n_b -= 1
        n_total -= 1
        probability *= p
    return probability, steps

print("Urn: 8 White + 4 Black balls — 3 draws without replacement")
print("=" * 58)

# Question 1: W1 B2 B3
p1, steps1 = multiplication_rule_balls(8, 4, ['W', 'B', 'B'])
print("\nQuestion 1: P(W₁ B₂ B₃) = ?  (Multiplication Rule)")
for i, (desc, p, frac) in enumerate(steps1, 1):
    print(f"  Step {i}: {desc} = {frac} = {p:.5f}")
frac_result = Fraction(8,12) * Fraction(4,11) * Fraction(3,10)
print(f"  P(W₁B₂B₃) = 8/12 × 4/11 × 3/10 = {frac_result} = {p1:.6f}")

print()

# Question 2: P(B₂) — black on 2nd draw
p_W1B2, _ = multiplication_rule_balls(8, 4, ['W', 'B'])
p_B1B2, _ = multiplication_rule_balls(8, 4, ['B', 'B'])
p_B2 = p_W1B2 + p_B1B2

f_W1B2 = Fraction(8,12) * Fraction(4,11)
f_B1B2 = Fraction(4,12) * Fraction(3,11)
f_B2   = f_W1B2 + f_B1B2

print("Question 2: P(B₂) = ?  (1st draw unknown)")
print(f"  P(W₁B₂) = 8/12 × 4/11 = {f_W1B2} = {p_W1B2:.6f}")
print(f"  P(B₁B₂) = 4/12 × 3/11 = {f_B1B2} = {p_B1B2:.6f}")
print(f"  P(B₂) = P(W₁B₂) + P(B₁B₂) = {f_B2} = {p_B2:.6f}")
print(f"  Note: P(B₂) = P(B₁) = 4/12 = {4/12:.6f}  (symmetry!)")

In [ ]:
# Chain rule — tree diagram visualization
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 7)
ax.axis('off')
ax.set_title('Multiplication Rule — Tree Diagram (8W + 4B, 3 Draws)',
             fontsize=13, fontweight='bold')

# Root
root_x, root_y = 0.3, 3.5
ax.text(root_x, root_y, 'Start\n(8W+4B)', ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#f0f0f0'), zorder=3)

# Level 1
branch1 = [('W', 1.5, 5.5, '8/12', '#92c5de'), ('B', 1.5, 1.5, '4/12', '#f4a582')]
for color, x, y, frac, col in branch1:
    ax.annotate('', xy=(x-0.2, y), xytext=(root_x+0.2, root_y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.3))
    ax.text(x, y, f'{color}₁', ha='center', va='center', fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.8), zorder=3)
    mid_x = (root_x + x) / 2 + 0.1
    mid_y = (root_y + y) / 2
    ax.text(mid_x, mid_y, frac, fontsize=8, color='navy', ha='center')

# Level 2
branch2 = [
    ('W', 1.5, 5.5, 'W', 2.8, 6.3, '7/11', '#92c5de'),
    ('W', 1.5, 5.5, 'B', 2.8, 4.7, '4/11', '#f4a582'),
    ('B', 1.5, 1.5, 'W', 2.8, 2.3, '8/11', '#92c5de'),
    ('B', 1.5, 1.5, 'B', 2.8, 0.7, '3/11', '#f4a582'),
]
for _, x1, y1, color2, x2, y2, frac2, col in branch2:
    ax.annotate('', xy=(x2-0.2, y2), xytext=(x1+0.2, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.1))
    ax.text(x2, y2, f'{color2}₂', ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.7), zorder=3)
    ax.text((x1+x2)/2+0.1, (y1+y2)/2, frac2, fontsize=7, color='navy', ha='center')

# Level 3 — highlight the W1B2B3 path
highlighted = [  # (parent_x, parent_y, color3, x3, y3, frac3, highlight)
    (2.8, 4.7, 'B', 4.2, 4.2, '3/10', True),
    (2.8, 0.7, 'B', 4.2, 0.3, '2/10', False),
    (2.8, 6.3, 'B', 4.2, 6.0, '4/10', False),
]
for x1, y1, color3, x2, y2, frac3, highlight in highlighted:
    style = '#ff7f00' if highlight else '#f4a582'
    lw    = 2.5       if highlight else 1.0
    ax.annotate('', xy=(x2-0.2, y2), xytext=(x1+0.2, y1),
                arrowprops=dict(arrowstyle='->', color=style, lw=lw))
    ax.text(x2, y2, f'{color3}₃', ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=style, alpha=0.8), zorder=3)
    ax.text((x1+x2)/2+0.1, (y1+y2)/2, frac3, fontsize=7, color='navy', ha='center')

# Highlighted path label
ax.text(4.8, 4.2, f'W₁B₂B₃\n= 8/12×4/11×3/10\n= {Fraction(8,12)*Fraction(4,11)*Fraction(3,10)}',
        ha='left', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#ffffcc', edgecolor='orange', lw=1.5))

plt.tight_layout()
plt.show()

---
## 3. The Law of Total Probability

If $A_1, \ldots, A_n$ are mutually exclusive and $\bigcup_{i=1}^n A_i = S$ (**a partition**), then:

$$P(B) = \sum_{j=1}^n P(B \mid A_j)\, P(A_j)$$

In [ ]:
def total_probability(p_Ai_list, p_BgAi_list):
    """P(B) = Σ P(B|Ai) * P(Ai)"""
    assert abs(sum(p_Ai_list) - 1.0) < 1e-9, "P(Ai) values must sum to 1!"
    return sum(prior * lik for prior, lik in zip(p_Ai_list, p_BgAi_list))

# ──────────────────────────────────────────
# Randomized Response Survey (drug use example)
print("=" * 55)
print("Randomized Response Survey — Drug Use Question")
print("=" * 55)

# A1 = {heads}, A2 = {tails}
p_A1 = 0.5  # P(heads)
p_A2 = 0.5  # P(tails)

# P(YES|heads) = P(carpooling) = 0.35
# P(YES|tails) = P(drug use) = ?
p_yes_gA1   = 0.35      # known carpooling rate
p_yes_obs   = 1420 / 8000  # observed YES rate

# Total probability: P(YES) = P(YES|A1)*P(A1) + P(drug)*P(A2)
# p_yes_obs = p_yes_gA1 * p_A1 + p_drug * p_A2
p_drug = (p_yes_obs - p_yes_gA1 * p_A1) / p_A2

print(f"\nObserved: {1420}/{8000} = {p_yes_obs:.5f} YES responses")
print(f"P(YES|heads) = P(carpooling) = {p_yes_gA1}")
print(f"P(YES) = P(YES|heads)×P(heads) + P(drug)×P(tails)")
print(f"{p_yes_obs:.5f} = {p_yes_gA1} × {p_A1} + P(drug) × {p_A2}")
print(f"P(drug) = ({p_yes_obs:.5f} - {p_yes_gA1*p_A1}) / {p_A2} = {p_drug:.4f}")
print(f"\nConclusion: Approximately {p_drug*100:.1f}% of employees use drugs")

# Verification
p_check = total_probability([p_A1, p_A2], [p_yes_gA1, p_drug])
print(f"\nVerification: P(YES) = {p_check:.5f} ≈ {p_yes_obs:.5f}  {'✓' if abs(p_check-p_yes_obs)<1e-6 else '✗'}")

In [ ]:
# Law of Total Probability visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Randomized survey — tree
ax = axes[0]
ax.set_xlim(-0.3, 3); ax.set_ylim(0, 4); ax.axis('off')
ax.set_title('Randomized Response — Law of Total Probability', fontsize=11, fontweight='bold')

# Root
ax.text(0.1, 2, 'Employee\nFlips a Coin', ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#f0f0f0'))

branches = [
    ('Heads (0.5)', 1.2, 3.2, 'Carpooling?', '#92c5de', 0.35, '#4daf4a', '#e41a1c'),
    ('Tails (0.5)', 1.2, 0.8, 'Do you use\ndrugs?', '#f4a582', p_drug, '#4daf4a', '#e41a1c'),
]
for label, x1, y1, question, col, p_yes, col_y, col_n in branches:
    ax.annotate('', xy=(x1-0.15, y1), xytext=(0.3, 2),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.3))
    ax.text(x1, y1+0.3, label, ha='center', fontsize=8, color='navy')
    ax.text(x1, y1, question, ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.7))
    ax.text(2.4, y1+0.3, f'YES ({p_yes:.3f})', ha='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col_y, alpha=0.6))
    ax.text(2.4, y1-0.3, f'NO ({1-p_yes:.3f})', ha='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col_n, alpha=0.4))
    ax.annotate('', xy=(2.1, y1+0.3), xytext=(x1+0.3, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9))
    ax.annotate('', xy=(2.1, y1-0.3), xytext=(x1+0.3, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9))

ax.text(1.5, 0.1, f'P(YES) = {p_check:.4f}', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Right: Partition concept
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 6); ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('Law of Total Probability: Partition of S', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.3, 0.3), 9.4, 5.2, fill=False, edgecolor='black', linewidth=2)
ax2.add_patch(rect)
partitions = [(0.3, 0.3, 2.5, 5.2, '#92c5de', 'A₁'), (2.8, 0.3, 3.5, 5.2, '#f4a582', 'A₂'),
              (6.3, 0.3, 2.0, 5.2, '#b2df8a', 'A₃'), (8.3, 0.3, 1.4, 5.2, '#984ea3', 'A₄')]
for x, y, w, h, col, lbl in partitions:
    r = plt.Rectangle((x, y), w, h, color=col, alpha=0.4)
    ax2.add_patch(r)
    ax2.text(x + w/2, y + h/2 + 1.0, lbl, ha='center', va='center',
             fontsize=13, fontweight='bold', color=col)

# Event B (horizontal ellipse)
from matplotlib.patches import Ellipse
ell = Ellipse((5, 2.9), 8, 1.8, color='gold', alpha=0.55, zorder=2)
ax2.add_patch(ell)
ax2.text(5, 2.9, 'B', ha='center', va='center', fontsize=14,
         fontweight='bold', color='darkgoldenrod', zorder=3)

ax2.text(5, 0.0, 'P(B) = Σ P(B|Aᵢ)·P(Aᵢ)',
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax2.text(9.5, 5.3, 'S', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Bayes' Theorem

If $A_1,\ldots,A_n$ form a partition, $P(A_i)>0$, and $P(B)>0$:

$$P(A_i \mid B) = \frac{P(B \mid A_i)\,P(A_i)}{\displaystyle\sum_{j=1}^n P(B \mid A_j)\,P(A_j)}$$

- $P(A_i)$: **prior** probability
- $P(A_i \mid B)$: **posterior** probability  
- Goal: **reverse** the conditioning — compute $P(A|B)$ when $P(B|A)$ is known

In [ ]:
def bayes(p_prior, p_likelihood):
    """
    p_prior      : [P(A1), P(A2), ..., P(An)]  — prior probabilities
    p_likelihood : [P(B|A1), ..., P(B|An)]     — likelihoods
    Returns      : [P(A1|B), ..., P(An|B)]      — posterior probabilities
    """
    p_B = sum(l * p for l, p in zip(p_likelihood, p_prior))
    posterior = [(l * p) / p_B for l, p in zip(p_likelihood, p_prior)]
    return posterior, p_B

# ──────────────────────────────────────────
# Vassar Example
print("Vassar Students — Bayes' Theorem")
print("=" * 50)

# Prior probabilities — department distribution
prior = [0.25, 0.55, 0.20]   # [math, music, economics]
dept  = ['Mathematics', 'Music', 'Economics']

# P(appointment | department)
p_appt = [0.90, 0.50, 0.10]

posterior, p_B = bayes(prior, p_appt)

print(f"{'Department':<14} {'P(Aᵢ)':>8} {'P(B|Aᵢ)':>10} {'P(B|Aᵢ)×P(Aᵢ)':>16} {'P(Aᵢ|B)':>10}")
print("-" * 62)
for d, pri, lik, post in zip(dept, prior, p_appt, posterior):
    joint = pri * lik
    print(f"{d:<14} {pri:>8.2f} {lik:>10.2f} {joint:>16.4f} {post:>10.4f}")
print("-" * 62)
print(f"{'P(B) = ':>42} {p_B:>10.4f}")
print()
print(f"Probability that a student who made an appointment is a math student:")
print(f"P(Mathematics|Appointment) = {posterior[0]:.4f} ≈ {posterior[0]*100:.1f}%")
print(f"\nPrior P(Mathematics) = {prior[0]:.2f} → Posterior P(Mathematics|Appointment) = {posterior[0]:.4f}")
print(f"The appointment news made mathematics more likely! (0.25 → {posterior[0]:.2f})")

In [ ]:
# Medical test example — practical importance of Bayes' Theorem
print("Medical Test Example — Bayes' Theorem")
print("=" * 50)
print("Disease prevalence: 1%  |  Test sensitivity: 99%  |  Test specificity: 95%")
print()

p_sick    = 0.01    # P(Sick)
p_healthy = 0.99    # P(Healthy)
p_pos_gsick    = 0.99  # P(Test+ | Sick)
p_pos_ghealthy = 0.05  # P(Test+ | Healthy) = 1 - specificity

prior_t = [p_sick, p_healthy]
lik_t   = [p_pos_gsick, p_pos_ghealthy]
post_t, p_test_pos = bayes(prior_t, lik_t)

print(f"P(Sick | Test+)     = {post_t[0]:.4f} = {post_t[0]*100:.1f}%")
print(f"P(Healthy | Test+)  = {post_t[1]:.4f} = {post_t[1]*100:.1f}%")
print(f"\nSurprising: Even after a positive test, the probability of being sick is only {post_t[0]*100:.1f}%!")
print(f"Why? Low prevalence ({p_sick*100}%) → false positives outnumber true positives")

# P(Sick|Test+) for various prevalence values
print()
print(f"{'Prevalence':>12} | {'P(Sick|Test+)':>16}")
print("-" * 32)
for prev in [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]:
    p_hlth = 1 - prev
    posts, _ = bayes([prev, p_hlth], [p_pos_gsick, p_pos_ghealthy])
    print(f"{prev*100:>10.1f}%  | {posts[0]*100:>14.2f}%")

In [ ]:
# Bayesian update visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Vassar — prior vs posterior
ax = axes[0]
x  = np.arange(len(dept))
w  = 0.35
b1 = ax.bar(x - w/2, prior,     w, label='Prior P(Aᵢ)',              color='#4C72B0', alpha=0.8, edgecolor='black', lw=0.5)
b2 = ax.bar(x + w/2, posterior, w, label='Posterior P(Aᵢ|Appointment)', color='#DD8452', alpha=0.8, edgecolor='black', lw=0.5)
for bar, v in zip(b1, prior):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
for bar, v in zip(b2, posterior):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(dept, fontsize=11)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('Vassar: Prior → Posterior (Bayesian Update)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 0.75)

# Right: Prevalence vs P(Sick|Test+)
ax2 = axes[1]
prevs = np.linspace(0.001, 0.5, 200)
post_sick = []
for prev in prevs:
    p_hlth = 1 - prev
    p_B_test = p_pos_gsick * prev + p_pos_ghealthy * p_hlth
    post_sick.append(p_pos_gsick * prev / p_B_test)

ax2.plot(prevs * 100, [p*100 for p in post_sick], color='#d62728', linewidth=2.5)
ax2.axhline(50, color='gray', linestyle='--', alpha=0.6, label='50% threshold')
ax2.scatter([1], [post_t[0]*100], color='red', s=120, zorder=5)
ax2.annotate(f'Prevalence=1%\nP(Sick|+)={post_t[0]*100:.1f}%',
             xy=(1, post_t[0]*100), xytext=(8, 20),
             arrowprops=dict(arrowstyle='->', color='black'),
             fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax2.set_xlabel('Disease Prevalence (%)', fontsize=11)
ax2.set_ylabel('P(Sick | Test+) (%)', fontsize=11)
ax2.set_title('Bayes: How Prevalence Affects P(Sick|Test+)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Independent Events

**Definition:** $A$ and $B$ are independent if:
$$P(A \cap B) = P(A) \times P(B)$$

Equivalent statement: $P(A \mid B) = P(A)$ — knowing $B$ does not affect $A$.

**Important:** Independent ≠ Mutually exclusive (disjoint)

In [ ]:
def independence_check(S_set, A_set, B_set, name_A="A", name_B="B"):
    """Check whether P(A∩B) = P(A)×P(B)"""
    n = len(S_set)
    p_A   = len(A_set) / n
    p_B   = len(B_set) / n
    p_AnB = len(A_set & B_set) / n
    p_prod = p_A * p_B
    independent = abs(p_AnB - p_prod) < 1e-9
    print(f"  P({name_A})={p_A:.4f}, P({name_B})={p_B:.4f}")
    print(f"  P({name_A}∩{name_B})={p_AnB:.4f},  P({name_A})×P({name_B})={p_prod:.4f}")
    print(f"  Independent? {'✓ YES' if independent else '✗ NO'}")
    return independent

S = {(i,j) for i in range(1,7) for j in range(1,7)}
print("=" * 52)
print("Independence Examples — Two Dice")
print("=" * 52)

# 1. Two independent die rolls — 1st even, 2nd even
A1 = {(i,j) for i,j in S if i % 2 == 0}
B1 = {(i,j) for i,j in S if j % 2 == 0}
print("\n1. A={1st die even}, B={2nd die even}")
independence_check(S, A1, B1, "1st_even", "2nd_even")

# 2. Sum = 7 and 1st die = 4 (also independent)
A2 = {(i,j) for i,j in S if i+j == 7}
B2 = {(i,j) for i,j in S if i == 4}
print("\n2. A={Sum=7}, B={1st die=4}")
independence_check(S, A2, B2, "Sum7", "1st=4")

# 3. Disjoint but dependent
A3 = {(i,j) for i,j in S if i == 1}
B3 = {(i,j) for i,j in S if i == 2}
print("\n3. A={1st die=1}, B={1st die=2} — DISJOINT (mutually exclusive)")
independence_check(S, A3, B3, "1st=1", "1st=2")
print(f"  → Disjoint events with P(A)>0, P(B)>0 are NEVER independent!")

print()
# Coin toss: 5 consecutive heads
p_5_heads = (1/2)**5
print(f"Application of independence: P(5 consecutive heads) = (1/2)⁵ = {p_5_heads:.5f} = {Fraction(1,32)}")

In [ ]:
# Verifying independence properties
print("Properties of Independence")
print("=" * 48)

# Single fair die
S_single = set(range(1, 7))
A = {2, 4, 6}    # even
B = {1, 2, 3}    # ≤ 3

Ac = S_single - A
Bc = S_single - B

def p_single(E): return len(E) / len(S_single)
def indep(X, Y): return abs(p_single(X&Y) - p_single(X)*p_single(Y)) < 1e-9

print(f"A = {sorted(A)} (even),  P(A) = {p_single(A):.4f}")
print(f"B = {sorted(B)} (≤3),    P(B) = {p_single(B):.4f}")
print()
print(f"A and B  independent?  {indep(A,B)}  → P(A∩B)={p_single(A&B):.4f}, P(A)P(B)={p_single(A)*p_single(B):.4f}")
print(f"A and Bᶜ independent?  {indep(A,Bc)} → P(A∩Bᶜ)={p_single(A&Bc):.4f}, P(A)P(Bᶜ)={p_single(A)*p_single(Bc):.4f}")
print(f"Aᶜ and B independent?  {indep(Ac,B)} → P(Aᶜ∩B)={p_single(Ac&B):.4f}, P(Aᶜ)P(B)={p_single(Ac)*p_single(B):.4f}")
print(f"Aᶜ and Bᶜ independent? {indep(Ac,Bc)} → P(Aᶜ∩Bᶜ)={p_single(Ac&Bc):.4f}, P(Aᶜ)P(Bᶜ)={p_single(Ac)*p_single(Bc):.4f}")
print()
print("Conclusion: If A and B are independent → A and Bᶜ, Aᶜ and B, Aᶜ and Bᶜ are also independent! ✓")

---
## 6. Mutual Independence

**Pairwise independent:** Every pair of events is independent: $P(A_i \cap A_j) = P(A_i)P(A_j)$

**Mutually independent:** Pairwise independence **and** $P(A \cap B \cap C) = P(A)P(B)P(C)$

**Pairwise independent ≠ Mutually independent!**

In [ ]:
# Two Dice: A={Sum=7}, B={First=4}, C={Second=3}
# Pairwise independent but NOT mutually independent

S2 = [(i,j) for i in range(1,7) for j in range(1,7)]
n2 = len(S2)

A = {(i,j) for i,j in S2 if i+j == 7}   # Sum = 7
B = {(i,j) for i,j in S2 if i == 4}      # First die = 4
C = {(i,j) for i,j in S2 if j == 3}      # Second die = 3

def p2(E): return len(E) / n2

print("Two Dice Example: A={Sum=7}, B={First=4}, C={Second=3}")
print("=" * 55)
print(f"P(A)={p2(A):.4f}={Fraction(len(A),n2)}, P(B)={p2(B):.4f}={Fraction(len(B),n2)}, P(C)={p2(C):.4f}={Fraction(len(C),n2)}")
print()

# Pairwise independence checks
pairs = [(A, B, 'A', 'B'), (B, C, 'B', 'C'), (A, C, 'A', 'C')]
print("Pairwise Independence:")
for X, Y, nx, ny in pairs:
    p_XY   = p2(X & Y)
    p_X_pY = p2(X) * p2(Y)
    equal  = abs(p_XY - p_X_pY) < 1e-9
    print(f"  P({nx}∩{ny}) = {p_XY:.4f} = {Fraction(len(X&Y),n2)},  "
          f"P({nx})×P({ny}) = {p_X_pY:.4f}  → {'✓ Independent' if equal else '✗ Dependent'}")

# Triple check
p_ABC   = p2(A & B & C)
p_A_B_C = p2(A) * p2(B) * p2(C)
print()
print("Triple Independence:")
print(f"  P(A∩B∩C) = {p_ABC:.4f} = {Fraction(len(A&B&C),n2)},  P(A)×P(B)×P(C) = {p_A_B_C:.4f} = {Fraction(1,216)}")
equal3 = abs(p_ABC - p_A_B_C) < 1e-9
print(f"  → {'✓ Mutually independent' if equal3 else '✗ NOT mutually independent'}")
print()
print("Why? A∩B∩C = {(4,3)} — Sum=7 AND First=4 AND Second=3 are all required simultaneously")
print("These three conditions are logically linked → mutual independence breaks down")

In [ ]:
# Application of independent events — system reliability
print("Application: System Reliability")
print("=" * 48)
print("Each component independently has P(working) = 0.95")

p_comp = 0.95

# Series connection: P(system) = Π P(component)
print("\n1. Series Connection (all components must work):")
for n in [1, 2, 3, 5, 10]:
    p_series = p_comp ** n
    print(f"   n={n:2d} components: P(system) = 0.95^{n} = {p_series:.5f}")

# Parallel connection: P(system fails) = Π P(component fails)
print("\n2. Parallel Connection (at least one must work):")
for n in [1, 2, 3, 5, 10]:
    p_parallel = 1 - (1 - p_comp) ** n
    print(f"   n={n:2d} components: P(system) = 1 - 0.05^{n} = {p_parallel:.7f}")

---
## 7. Probability Perception and Cognitive Biases — The Linda Problem

**Conjunction fallacy:** People tend to believe $P(A \cap B) > P(B)$.
Yet the mathematical truth is: $P(A \cap B) \leq P(B)$ always!

Discovered by Tversky and Kahneman (1983).

In [ ]:
# The Linda Problem — mathematical analysis
print("The Linda Problem — Mathematical Analysis")
print("=" * 52)

# Hypothetical probabilities
p_A  = 0.70   # P(Active in feminist movement)
p_B  = 0.15   # P(Bank teller)

# P(A∩B) under various scenarios

print(f"A = {{Linda is active in the feminist movement}},  P(A) = {p_A}")
print(f"B = {{Linda is a bank teller}},                   P(B) = {p_B}")
print()
print(f"MATHEMATICAL CONSTRAINT: P(A∩B) ≤ min(P(A), P(B)) = {min(p_A, p_B)}")
print()

# Assuming independence (upper bound scenario)
p_AnB_indep = p_A * p_B
print(f"Under independence assumption:  P(A∩B) = {p_AnB_indep:.4f}")
print(f"Always: P(A∩B)={p_AnB_indep:.4f} ≤ P(B)={p_B:.4f}  → {'✓' if p_AnB_indep <= p_B else '✗'}")
print()

# Numerical results from the Kahneman-Tversky experiment
print("Historical Experimental Results (Tversky & Kahneman, 1983):")
print("-" * 55)
results = [
    ('Linda is active in the feminist movement', 'A',   2.1),
    ('Linda is a bank teller and a feminist',    'A∩B', 4.1),
    ('Linda is a bank teller',                   'B',   6.2),
]
print(f"{'Statement':<48} {'Event':>5} {'Avg. Rank':>10}")
print("-" * 65)
for statement, event, rank in results:
    marker = " ← LOWER rank = MORE probable" if event == 'A∩B' else ""
    print(f"{statement:<48} {event:>5} {rank:>10.1f}{marker}")
print()
print("PROBLEM: 90% of participants rated 'bank teller AND feminist'")
print("         as more probable than 'bank teller' alone.")
print(f"         Yet: P(A∩B) ≤ P(B) ALWAYS!")

In [ ]:
# Linda problem visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mathematical truth — P(A∩B) ≤ P(A), P(A∩B) ≤ P(B)
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Mathematical Truth: P(A∩B) ≤ P(B)', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.2, 0.5), 9.6, 5, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(rect)
cA = Circle((3.8, 3.0), 2.2, color='#92c5de', alpha=0.55, zorder=2)
cB = Circle((6.5, 3.0), 2.0, color='#f4a582', alpha=0.55, zorder=2)
for c in [cA, cB]: ax.add_patch(c)
ax.add_patch(Circle((3.8,3.0),2.2,fill=False,ec='#2166ac',lw=2,zorder=3))
ax.add_patch(Circle((6.5,3.0),2.0,fill=False,ec='#ca0020',lw=2,zorder=3))
ax.text(2.5, 3.8, 'A\n(Feminist)', ha='center', fontsize=10, fontweight='bold', color='#2166ac')
ax.text(7.8, 3.8, 'B\n(Bank teller)', ha='center', fontsize=10, fontweight='bold', color='#ca0020')
ax.text(5.2, 3.0, 'A∩B', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(5, 0.2,
        'A∩B ⊆ B  →  P(A∩B) ≤ P(B)',
        ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', lw=1.5))
ax.text(9.5, 5.3, 'S', fontsize=12, fontweight='bold')

# Right: Comparison of experimental results
ax2 = axes[1]
labels     = ['A\n(Feminist)', 'A∩B\n(Feminist\n+Bank teller)', 'B\n(Bank teller)']
ranks_exp  = [2.1, 4.1, 6.2]
ranks_math = [2.1, 6.5, 6.2]  # Mathematically correct order (A∩B least probable)

x = np.arange(len(labels))
w = 0.35
b1 = ax2.bar(x - w/2, ranks_exp,  w, label='Experiment (human)',
             color=['#4daf4a','#e41a1c','#984ea3'], alpha=0.85, edgecolor='black', lw=0.5)
b2 = ax2.bar(x + w/2, ranks_math, w, label='Mathematical ordering',
             color=['#4daf4a','#aaaaaa','#984ea3'], alpha=0.5, edgecolor='black', lw=0.5,
             hatch='///')

for bar, v in zip(b1, ranks_exp):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}', ha='center', va='bottom', fontsize=10)

ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel('Average Rank (1=most probable, 8=least probable)', fontsize=9)
ax2.set_title('Linda Problem: Experiment vs Mathematics\n(Lower rank = More probable)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 8)
ax2.grid(True, alpha=0.3, axis='y')
ax2.annotate('CONJUNCTION\nFALLACY!', xy=(1-w/2+0.05, 4.1), xytext=(1.8, 6.5),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Summary

In [ ]:
print("=" * 68)
print("CHAPTER 3 SUMMARY — Conditional Probability and Independence")
print("=" * 68)

summary = [
    ("Conditional Probability", "P(A|B) = P(A∩B) / P(B)"),
    ("Intuition",               "P(A|B) = |A∩B| / |B|  (equally likely)"),
    ("Proposition 1",           "P(A|A) = 1"),
    ("Proposition 2",           "P(Aᶜ|B) = 1 − P(A|B)"),
    ("Multiplication Rule",     "P(A∩B) = P(A|B)·P(B)"),
    ("Chain Rule",              "P(∩Aᵢ) = P(A₁)·P(A₂|A₁)·P(A₃|A₁,A₂)…"),
    ("Law of Total Prob.",      "P(B) = Σ P(B|Aᵢ)·P(Aᵢ)  (partition)"),
    ("Bayes' Theorem",          "P(Aᵢ|B) = P(B|Aᵢ)P(Aᵢ) / P(B)"),
    ("Prior / Posterior",       "P(Aᵢ): prior,  P(Aᵢ|B): posterior"),
    ("Independence",            "P(A∩B) = P(A)·P(B)  ↔  P(A|B) = P(A)"),
    ("Indep. ≠ Disjoint",      "Disjoint events are generally dependent!"),
    ("Mutual Independence",     "Pairwise indep. AND P(A∩B∩C)=P(A)P(B)P(C)"),
    ("Linda Problem",           "P(A∩B) ≤ P(B) always — conjunction fallacy!"),
]

print(f"{'Concept':<26} | {'Formula / Description'}")
print("-" * 68)
for concept, description in summary:
    print(f"{concept:<26} | {description}")
print("=" * 68)

---
## 9. Exercises

In [ ]:
# Problem 1: Two dice — sum ≥ 10, given first die ≥ 4
print("PROBLEM 1: P(Sum ≥ 10 | First die ≥ 4)")
S2 = {(i,j) for i in range(1,7) for j in range(1,7)}
A_p1 = {(i,j) for i,j in S2 if i+j >= 10}
B_p1 = {(i,j) for i,j in S2 if i >= 4}
ans1 = conditional_uniform(S2, A_p1, B_p1)
print(f"  |A∩B| = {len(A_p1&B_p1)},  |B| = {len(B_p1)},  P(A|B) = {ans1:.4f} = {Fraction(len(A_p1&B_p1),len(B_p1))}")

print()
# Problem 2: Law of total probability
print("PROBLEM 2: Factory production — Machine A: 60% output, 3% defective")
print("           Machine B: 40% output, 5% defective. P(random item is defective)?")
ans2 = total_probability([0.60, 0.40], [0.03, 0.05])
print(f"  P(Defective) = 0.60×0.03 + 0.40×0.05 = {0.60*0.03:.4f} + {0.40*0.05:.4f} = {ans2:.4f}")

print()
# Problem 3: Bayes
print("PROBLEM 3: Given a defective item, what is the probability it came from Machine A?")
post_p3, _ = bayes([0.60, 0.40], [0.03, 0.05])
print(f"  P(Machine A | Defective) = {post_p3[0]:.4f} ≈ {post_p3[0]*100:.1f}%")

print()
# Problem 4: Independence
print("PROBLEM 4: P(A)=0.4, P(B)=0.3, P(A∩B)=0.12 — are A and B independent?")
pa, pb, panb = 0.4, 0.3, 0.12
print(f"  P(A)×P(B) = {pa*pb:.4f},  P(A∩B) = {panb:.4f}  → {'✓ Independent' if abs(pa*pb-panb)<1e-9 else '✗ Dependent'}")